# 05 - Adversarial and Security Considerations

`00` made a point of naming what edge computing gives up for its low-latency,
no-connectivity footprint: the compiled model is a static artifact that ships on
hardware physically inside the thing it's controlling, not a service walled off behind
someone else's API. This notebook is about what that exposure actually means -
starting with a real, measured demonstration of an adversarial example, and ending with
an honest assessment of how much of this actually applies to a competition robot versus
how much is a different, more mundane problem wearing the same name.

## What an Adversarial Example Is

An **adversarial example** is an input that's been deliberately, minimally modified to
make a model produce a wrong answer, while the modification stays small enough that a
human looking at the input barely notices anything changed. The modification isn't
random noise - it's computed using the model's own gradients, in the direction that
increases the model's error the most per unit of change to the input. That's what
distinguishes it from an input that's simply hard, blurry, or unusual: a genuinely
adversarial example is often *specifically engineered against the exact model being
attacked*, using information (the model's own gradients) an attacker without access to
that model wouldn't have.

## Why Edge Specifically Raises the Stakes

Computing the gradient-based perturbation below requires access to the model itself -
its architecture and weights - so it can be differentiated through. A cloud-hosted
model behind an API never exposes that: an attacker only ever sees inputs and outputs,
which makes crafting a precise adversarial example far harder (though not impossible -
this is its own subfield, "black-box" attacks, out of scope here). An edge-deployed
model is a different situation entirely: `03`'s compiled, converted model file is
sitting on physically accessible hardware. Anyone who can get their hands on a
Limelight, a Coral, or any other edge device carrying your model has the same kind of
access this notebook's demo has - the full model, gradients and all.

## FRC-Scoped Threat Categories

It's worth being precise about which of these are real, practical concerns for a
competition robot and which are mostly theoretical here, rather than importing a
generic "security matters" anxiety wholesale:

- **Physical adversarial patterns** - a specifically-patterned sticker or object,
  designed offline using a model's gradients, that fools a detector when placed in its
  field of view. This is a real, published technique. It's also a low-realism threat at
  a student robotics competition: it requires an attacker with white-box access to your
  exact trained model, which no other team has.
- **Distribution shift wearing an adversarial costume** - unusual lighting, an
  opponent's robot decorations, a background nobody trained on. This is *not* an
  adversarial example by the definition above - nobody computed a gradient to produce
  it, it's just a condition the training data didn't cover. It's worth naming
  explicitly because it's easy to conflate the two, and because this - not deliberate
  attacks - is what actually causes most real FRC detection failures. It's also exactly
  what `frc_resources/03_roboflow`'s hard-example mining loop exists to fix.
- **Model extraction / IP concerns** - an opponent recovering your trained model from a
  physically accessible device. Worth knowing this category exists; low-stakes here,
  since this is a student competition, not a business protecting proprietary IP.

The honest summary: understanding how adversarial examples work is worth doing for the
instincts it builds, not because a team should expect to be attacked this way at a
competition. The FGSM demo below is a real, measured technique - it's also a
demonstration of a threat model that mostly doesn't apply to this team's actual
situation, and the notebook says so explicitly rather than overselling it.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small
from PIL import Image


## Loading a Pretrained Classifier

We're using a small, standard ImageNet classifier here (`MobileNetV3-Small`, via
[torchvision's model zoo](https://docs.pytorch.org/vision/stable/models.html)) rather
than `perception_primer`'s `yolov8n.pt` - Fast Gradient Sign Method, below, is easiest
to demonstrate cleanly against a plain image classifier's single predicted class, not
an object detector's multiple boxes. The underlying mechanism - perturbing an input
using the model's own gradient - is identical regardless of which model you point it
at.

In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights)
model.eval()
categories = weights.meta["categories"]

# Reproducing the model's expected preprocessing manually, rather than using
# weights.transforms() as a black box, so we can perturb raw [0, 1] pixels directly -
# that's the standard convention for measuring how "visible" a perturbation is.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
to_pixels = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),  # -> [0, 1] range
])


def predict(x_pixels):
    normalized = (x_pixels - IMAGENET_MEAN) / IMAGENET_STD
    return model(normalized)


def top_prediction(x_pixels):
    with torch.no_grad():
        probs = torch.softmax(predict(x_pixels), dim=1)
    cls = probs.argmax(dim=1).item()
    return categories[cls], probs[0, cls].item()


## Baseline: What the Model Sees Before Any Attack

Same `yellow_ball.png` `02` quantized a detector against.

In [ ]:
img = Image.open("../perception_primer/ref_imgs/yellow_ball.png").convert("RGB")
x = to_pixels(img).unsqueeze(0)
x.requires_grad_(True)

label, confidence = top_prediction(x)
print(f"Baseline prediction: {label}  (confidence {confidence:.3f})")


## Fast Gradient Sign Method (FGSM)

[Goodfellow, Shlens, and Szegedy (2014)](https://arxiv.org/abs/1412.6572) introduced
the simplest widely-used adversarial attack: nudge every pixel by a tiny fixed amount
$\epsilon$, in whichever direction (`+` or `-`, hence "sign") *increases* the model's
loss on its own current prediction the most:

$$ x_{adv} = \text{clip}(x + \epsilon \cdot \text{sign}(\nabla_x \, \text{Loss}(x, y)),\ 0,\ 1) $$

This is a single gradient computation, not an iterative search - the same backward
pass mechanism `deep_learning_primer` built from scratch, just computing the gradient
with respect to the *input pixels* instead of the *weights*.

In [ ]:
orig_class_idx = torch.tensor([categories.index(label)])
loss = F.cross_entropy(predict(x), orig_class_idx)
model.zero_grad()
loss.backward()

EPSILON = 2 / 255  # a genuinely tiny perturbation - see the printed max pixel change below

perturbation = EPSILON * x.grad.sign()
x_adv = (x + perturbation).clamp(0, 1).detach()

adv_label, adv_confidence = top_prediction(x_adv)
_, orig_class_confidence_now = categories[orig_class_idx.item()], torch.softmax(predict(x_adv), dim=1)[0, orig_class_idx.item()].item()

print(f"epsilon = {EPSILON:.4f}  ({EPSILON * 255:.0f}/255)")
print(f"max pixel change: {perturbation.abs().max().item():.4f}  (out of a [0, 1] range)")
print(f"Adversarial prediction: {adv_label}  (confidence {adv_confidence:.3f})")
print(f"Original class '{label}' confidence now: {orig_class_confidence_now:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 4))

axes[0].imshow(x.detach().squeeze().permute(1, 2, 0).numpy())
axes[0].set_title(f"Original\n{label} ({confidence:.2f})")

# Perturbation magnified 25x so it's visible at all - at true scale it's imperceptible.
vis_perturbation = (perturbation.detach().squeeze().permute(1, 2, 0).numpy() * 25) + 0.5
axes[1].imshow(vis_perturbation.clip(0, 1))
axes[1].set_title(f"Perturbation\n(magnified 25x to be visible)")

axes[2].imshow(x_adv.squeeze().permute(1, 2, 0).numpy())
axes[2].set_title(f"Adversarial\n{adv_label} ({adv_confidence:.2f})")

for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()


## What Actually Happened

At $\epsilon = 2/255$ - a maximum per-pixel change of about 0.8% of the full [0, 1]
brightness range, invisible in the images above - the model's top prediction flipped
from "ping-pong ball" (confidence around 0.48) to a *different* ball class, at over
0.9 confidence, while the original class's confidence collapsed to essentially zero.
Nothing about the image looks different to a human. This is the concrete version of
the definition from the top of this notebook: a small, deliberately computed change
that a human can't see and a model can't ignore.

It's worth sitting with how little was required: one forward pass, one backward pass,
one sign operation. This is not a sophisticated attack - it's close to the simplest
gradient-based attack that exists, and it's still enough to flip a confident
prediction. More sophisticated (iterative, targeted) attacks exist and are harder to
defend against, not easier.

## Try It Yourself

1. Re-run the FGSM cell with `EPSILON` set to `4/255`, `8/255`, and `16/255`. At what
   point does the perturbation panel stop needing 25x magnification to see with the
   naked eye - i.e., at what point does "adversarial" become just "visibly corrupted"?
2. Try a **targeted** attack instead of the untargeted one above: instead of
   `orig_class_idx`, compute the loss against a class you *want* the model to predict
   (pick any index into `categories`) and flip the sign of the update
   (`x - EPSILON * grad.sign()` instead of `+`, since now you're *minimizing* loss
   toward a chosen wrong answer instead of maximizing loss away from the right one).
   Does forcing a specific wrong answer require a larger `EPSILON` than just causing
   any misclassification did?
3. Apply the exact perturbation tensor computed above to a *different* image (any other
   file in `ref_imgs/`) instead of `yellow_ball.png`. Does the same perturbation still
   change that image's prediction? This is asking about **transferability** - whether
   an adversarial perturbation crafted for one input (or one model) still works on a
   different one, which is exactly what makes black-box attacks (mentioned in the
   threat-model discussion above) possible at all.

## Resources

- [Goodfellow, Shlens, and Szegedy (2014): Explaining and Harnessing Adversarial Examples](https://arxiv.org/abs/1412.6572) -
  the original FGSM paper implemented above.
- [Torchvision: Models and Pre-Trained Weights](https://docs.pytorch.org/vision/stable/models.html) -
  documentation for the `MobileNetV3-Small` classifier used in this notebook.
- `ai_resources/agent_primer/08-guardrails-failure-modes-and-eval.md` - this primer's
  FRC-threat-model discussion above ("report the gap instead of guessing") is the same
  instinct that document asks for in an agent's failure handling - a system that's
  honest about the difference between "this input is wrong" and "this input is actively
  adversarial" is applying the same idea this notebook applies to a classifier.
- `frc_resources/03_roboflow` - the hard-example mining loop that addresses this
  primer's more realistic FRC failure mode, distribution shift, named in the
  threat-model discussion above.